# 02 · F5 최종 평가: 초기 R0(F0)·F4 대비 test·RouterArena
F5 최종 모델(seed 3개)을 잠근 뒤 초기 R0(운영 1.0, F0)와 함께 **봉인 test 7,202문항**과 **RouterArena 8,379문항**을 8192 토큰으로 평가합니다. 결과를 보고 다시 학습하지 않습니다.

- F4 run의 최종 예측을 해시 검증 후 불러와 같은 문항에서 F0·F4·F5를 비교하고 F5−F4 차이도 냅니다. RouterArena에는 F1~F3 최종 예측도 함께 들어갑니다.
- test: 원래 난이도 라벨 기준 정확도·macro-F1, 라우팅 첫 시도 품질·비용, 혼합선 대비 이득 / RouterArena: 공식 라벨 일치율·macro-F1
- 차이는 질문 그룹 bootstrap 95% 구간으로 보고합니다.

## 공통 준비
개인 `alexyoo0506@gmail.com` Colab Pro와 해당 계정의 Drive를 사용합니다. Drive 연결 시 해당 계정을 선택하고 Google 화면에서 직접 승인합니다. GCP·Vertex 제출, GCS 업로드, 운영 `CURRENT` 변경 코드는 없습니다.

제공한 실행 패키지의 `retraining-effect-f5` 폴더를 `MyDrive/LLM_Router/` 아래에 업로드하세요. 봉인 test(`inputs/test.parquet`)는 이 폴더에 들어 있습니다. 첫 실험 폴더 `retraining-effect`의 로거 결과·run001, F3 폴더 `retraining-effect-f3`, F4 폴더 `retraining-effect-f4`는 읽기 전용으로 재사용합니다. 입력 데이터와 R0(운영 1.0) 모델은 **R1→R0 전이 실험 때 올린 `MyDrive/LLM_Router/transfer-experiments/inputs/experiment-A`를 읽기 전용으로 재사용**합니다. 경로·SHA가 모두 채워져 있으므로 수정 없이 전체 실행합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

BASE = Path("/content/drive/MyDrive/LLM_Router/retraining-effect-f5")
FIRST = Path("/content/drive/MyDrive/LLM_Router/retraining-effect")  # 첫 실험: 로거·F1/F2 결과 재사용(읽기 전용)
F3_BASE = Path("/content/drive/MyDrive/LLM_Router/retraining-effect-f3")  # F3 결과(있으면 비교에만 사용)
F4_BASE = Path("/content/drive/MyDrive/LLM_Router/retraining-effect-f4")  # F4 결과(test·RouterArena 직접 비교)
SOURCE_ZIP = BASE / "source" / "retraining-source-f5.zip"
EXPECTED_SOURCE_SHA256 = SOURCE_ZIP.with_suffix(".sha256").read_text().split()[0]
RUN_ID = "retraining-effect-v4-f5-run001"
RUN = BASE / "runs" / RUN_ID
LOG = FIRST / "logger" / "logger-log.parquet"
REFERENCES = [FIRST / "runs" / "retraining-effect-v1-run001", F3_BASE / "runs" / "retraining-effect-v2-f3-run001",
              F4_BASE / "runs" / "retraining-effect-v3-f4-run001"]
EXTRA = BASE / "inputs" / "extra-inputs.json"  # 봉인 test 7,202문항(이 kit에 포함)
# R1->R0 전이 실험 때 올린 입력과 복구된 anchor 모델을 읽기 전용으로 재사용한다.
INPUTS = Path("/content/drive/MyDrive/LLM_Router/transfer-experiments/inputs/experiment-A")
PINS = INPUTS / "source_pins.json"
DATA = INPUTS / "data"

assert len(EXPECTED_SOURCE_SHA256) == 64, "소스 ZIP의 SHA-256 파일이 필요합니다"
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
REPO = Path("/content/router-retraining-source")
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, expected in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, name
EXPERIMENT = REPO / "ml/notebooks/retraining_effect"
HELPER = EXPERIMENT / "experiment.py"
CONFIG = EXPERIMENT / "experiment_config_f5.json"
ENV = dict(os.environ, PYTHONPATH=str(REPO / "ml/src"))
def run(*args):
    subprocess.run([sys.executable, str(HELPER), *map(str, args)], env=ENV, check=True)
assert PINS.is_file(), f"전이 실험 입력이 필요합니다: {PINS}"
assert EXTRA.is_file(), f"kit의 봉인 test 입력이 필요합니다: {EXTRA}"
assert LOG.is_file() and (REFERENCES[0] / "plan.json").is_file(), "첫 실험 retraining-effect 폴더가 필요합니다"
print("Run directory:", RUN)


## 패키지 설치
Colab 메뉴 **런타임 → 런타임 유형 변경 → 런타임 버전 2026.07**(Python 3.12.13), GPU **T4**를 선택합니다. 설치 후 이미 import한 패키지 버전이 다르면 세션을 재시작하고 경로 설정 셀부터 다시 실행하세요.

In [ ]:
assert sys.version_info[:2] in {(3, 11), (3, 12)}, (f"현재 Python {sys.version.split()[0]}. 런타임 버전을 2026.07(Python 3.12)로 바꾸고 처음부터 실행하세요.")
# Text-only model: remove preinstalled optional packages incompatible with pinned torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(EXPERIMENT / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO / "ml")], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU(T4) 런타임을 선택하세요"
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)

In [ ]:
if not (RUN / "final-lock.json").exists():
    run("final-lock", "--run-dir", RUN)
for role in ["test", "routerarena"]:
    run("score-final", "--run-dir", RUN, "--arm", "f0_frozen", "--seed", 0, "--role", role)
    for seed in [42, 43, 44]:
        run("score-final", "--run-dir", RUN, "--arm", "f5_matched_prior", "--seed", seed, "--role", role)
references = []
for ref in REFERENCES:
    if (ref / "plan.json").is_file():
        references += ["--reference-run", ref]
    else:
        print("참고 run 없음(건너뜀):", ref)
run("compare-final", "--run-dir", RUN, *references)
from IPython.display import display, Image, Markdown
import pandas as pd
report = RUN / "report-final"
display(pd.read_csv(report / "paired-comparisons.csv"))
display(pd.read_csv(report / "test-routing-by-source.csv"))
display(Image(filename=str(report / "test-quality-cost.png")))
display(Markdown((report / "REPORT.md").read_text()))


## 결과 하나만 전달
아래 셀이 모델 가중치를 제외한 보고서·예측·실행 기록을 `results-for-review.zip` 하나로 내려받습니다.

In [ ]:
review_zip = RUN / "results-for-review.zip"
with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in ["plan.json", "plan.sha256.json", "final-lock.json"]:
        z.write(RUN / name, name)
    for folder in ["report-final", "scores"]:
        for path in sorted((RUN / folder).rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RUN))
    for path in sorted((RUN / "arms").rglob("*.json")):
        relative = path.relative_to(RUN)
        if "checkpoints" not in relative.parts and path.name in {"completed.json", "stage-contract.json", "metrics.json"}:
            z.write(path, relative)
print("최종 검토용 파일:", review_zip)
from google.colab import files
files.download(str(review_zip))
